# 02 — Create train/test batches and completed rCLR matrices

Subjects are split into repeated train/test batches so all samples from one
subject remain together.

Feature filtering is learned from training samples only:
- relative abundance >= 0.01
- prevalence >= 0.10

rCLR is calculated without a pseudocount. Original zeros are missing during the
log-ratio calculation, then completed by iterative low-rank SVD so TEMPTED and
MEFISTO receive the same finite transformed matrix.

Filtered raw counts are retained for abundance-based plots.


In [1]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

NUMBER_OF_BATCHES = 10
TRAIN_FRACTION = 0.70
MINIMUM_PREVALENCE = 0.10
MINIMUM_RELATIVE_ABUNDANCE = 0.01
RCLR_COMPLETION_RANK = 5
RCLR_COMPLETION_ITERATIONS = 25
RANDOM_SEED = 7319

root = Path(".") if Path("data").exists() else Path("..")
dataset = sorted(
    path for path in (root / "data" / "processed_16s").iterdir()
    if path.is_dir()
    and (path / "counts.csv").exists()
    and (path / "metadata.csv").exists()
)[-1]

output = root / "data" / "splits" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

counts = pd.read_csv(dataset / "counts.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(
    dataset / "metadata.csv",
    dtype={"sample_id": str, "subject_id": str, "label": str},
)
counts = counts.loc[metadata["sample_id"]]
subjects = metadata.drop_duplicates("subject_id")[["subject_id", "label"]]

print("Input:", dataset)
print("Output:", output)


## TEMPTED preprocessing: `+0.5` then CLR

The TEMPTED paper applies CLR after adding a pseudocount of `0.5` to every microbiome count. Shi et al. justify `0.5` using a Dirichlet-multinomial bias argument and report that `0.5` and `1` give very similar performance, whereas `0.1` performs somewhat worse.

**Reference:** Shi P, Martino C, Han R, et al. *TEMPTED: time-informed dimensionality reduction for longitudinal microbiome studies.* **Genome Biology** 25, 317 (2024). https://doi.org/10.1186/s13059-024-03453-x

MEFISTO does **not** use this pseudocount in this workflow; its separate rCLR matrix leaves original zeros missing.


In [2]:
def relative_abundance(x):
    return x.div(x.sum(axis=1), axis=0)


def rclr(x):
    values = x.to_numpy(float)
    transformed = np.full(values.shape, np.nan)

    for row_number, row in enumerate(values):
        positive = row > 0
        logged = np.log(row[positive])
        transformed[row_number, positive] = logged - logged.mean()

    return pd.DataFrame(transformed, index=x.index, columns=x.columns)


def complete_rclr(train_rclr, test_rclr, rank=5, iterations=25):
    combined = pd.concat([train_rclr, test_rclr])
    missing = combined.isna().to_numpy()
    values = combined.to_numpy(float)

    column_means = np.nanmean(values, axis=0)
    column_means[~np.isfinite(column_means)] = 0
    filled = np.where(missing, column_means, values)

    rank = min(rank, max(1, min(filled.shape) - 1))

    for _ in range(iterations):
        row_means = filled.mean(axis=1, keepdims=True)
        centered = filled - row_means
        u, s, vt = np.linalg.svd(centered, full_matrices=False)
        reconstructed = (u[:, :rank] * s[:rank]) @ vt[:rank] + row_means
        filled[missing] = reconstructed[missing]

    completed = pd.DataFrame(filled, index=combined.index, columns=combined.columns)
    return completed.loc[train_rclr.index], completed.loc[test_rclr.index]


In [3]:
rows = []

for number in range(1, NUMBER_OF_BATCHES + 1):
    train_subjects, test_subjects = train_test_split(
        subjects,
        train_size=TRAIN_FRACTION,
        stratify=subjects["label"],
        random_state=RANDOM_SEED + number,
    )

    train_meta = metadata[metadata["subject_id"].isin(train_subjects["subject_id"])].copy()
    test_meta = metadata[metadata["subject_id"].isin(test_subjects["subject_id"])].copy()
    train_counts = counts.loc[train_meta["sample_id"]]
    test_counts = counts.loc[test_meta["sample_id"]]

    keep = (
        relative_abundance(train_counts) >= MINIMUM_RELATIVE_ABUNDANCE
    ).mean(axis=0) >= MINIMUM_PREVALENCE

    features = train_counts.columns[keep]
    train_counts = train_counts[features]
    test_counts = test_counts[features]

    train_rclr_raw = rclr(train_counts)
    test_rclr_raw = rclr(test_counts)
    train_rclr, test_rclr = complete_rclr(
        train_rclr_raw,
        test_rclr_raw,
        rank=RCLR_COMPLETION_RANK,
        iterations=RCLR_COMPLETION_ITERATIONS,
    )

    folder = output / f"batch_{number:03d}"
    folder.mkdir()

    for name, table in {
        "train_counts": train_counts,
        "test_counts": test_counts,
        "train_rclr": train_rclr,
        "test_rclr": test_rclr,
    }.items():
        table.rename_axis("sample_id").reset_index().to_csv(
            folder / f"{name}.csv.gz",
            index=False,
        )

    train_meta.to_csv(folder / "train_metadata.csv.gz", index=False)
    test_meta.to_csv(folder / "test_metadata.csv.gz", index=False)

    rows.append([
        folder.name,
        train_meta["subject_id"].nunique(),
        test_meta["subject_id"].nunique(),
        len(features),
        int(train_rclr_raw.isna().sum().sum()),
        int(test_rclr_raw.isna().sum().sum()),
    ])

summary = pd.DataFrame(
    rows,
    columns=[
        "batch",
        "train_subjects",
        "test_subjects",
        "features",
        "zeros_handled_train",
        "zeros_handled_test",
    ],
)
summary.to_csv(output / "batch_summary.csv", index=False)

print("Saved:", output)
summary


Saved: ../data/splits/20260806_210520


,batch,train_subjects,test_subjects,features,missing_train_rclr,missing_test_rclr
0,batch_001,146,63,45,4475,2194
1,batch_002,146,63,42,3748,1748
2,batch_003,146,63,43,4735,2280
3,batch_004,146,63,42,4016,2050
4,batch_005,146,63,41,4101,1542
5,batch_006,146,63,44,4784,2018
6,batch_007,146,63,44,4364,1970
7,batch_008,146,63,43,4587,1835
8,batch_009,146,63,41,4356,1651
9,batch_010,146,63,43,4146,2129
